# EXP-004 — Loss neo text cho head (seed 1)
Train từ cache như EXP-003, thêm `λ · L_anchor`. `L_anchor` giữ phân phối softmax ảnh → ngân hàng câu text (CLIP gốc) sau khi qua head, để head không lệch phía text.
Ngân hàng **A**: 336 câu mô tả hành động mặt, **không có tên lớp hay template của test**.

1. Tạo ngân hàng text (1 lần, lưu Drive).
2. Train λ ∈ {0.1, 1, 10}, mỗi λ 30 epoch, giữ head ở epoch 5 / 10 / 20 / 30.
3. Test từng mốc, so với EXP-000 seed1 và EXP-003 (λ = 0, cùng mốc epoch).

Mục tiêu: ở epoch 30, RAFDB không tụt (EXP-003: −5.2), FERPlus vẫn giữ phần lớn +4.3.

# Cấu hình

In [1]:
SEED = 1
EPOCHS = 30
SAVE_EPOCHS = [5, 10, 20, 30]         # test head ở các mốc này
LR = 1e-3
LAMBDAS = [0.1, 1, 10]                # trọng số loss neo
ANCHOR = "cross"                      # cross: KL ảnh→text bank | text: MSE ma trận cos giữa các câu
BANK_VARIANT = "A"                    # A: không tên lớp | B: thêm từ cảm xúc (ablation)
job_of = lambda lam: f"exp004_anchor_{ANCHOR}{BANK_VARIANT}_l{lam:g}_l14_seed{SEED}"
BRANCH = "exp/004-text-anchor-loss"

INSTRUCTION = "Please play the role of a facial action describer. Objectively describe the detailed facial actions of the person in the image."
TRAIN_DATA = "/content/CAER-S_Face/train"
FACE_ZIP = "/content/drive/MyDrive/Datasets/CAERS_Face_train.zip"
TEST_ZIP = "/content/drive/MyDrive/Datasets/static_test_sets.zip"
PRE_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp001/full"     # target đã cache ở EXP-001 (targets/chunk_*.npz)
CLIP_CACHE = f"{PRE_DIR}/clip_feats_l14.npz"                       # feature CLIP, tính 1 lần
OUT_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp004"
BANK = f"{OUT_DIR}/text_bank_{BANK_VARIANT}_l14.npz"
EXP003_CSV = "/content/drive/MyDrive/ExpCLIP_output/exp003/exp003_cache_l14_seed1-results.csv"   # λ = 0

import os, subprocess
def run(cmd):
    """Chạy lệnh shell, in output trực tiếp; lỗi thì dừng cell (khác với `!`)."""
    # bash + pipefail: lệnh có `| tee` vẫn báo lỗi nếu python lỗi
    p = subprocess.Popen("set -o pipefail; " + cmd, shell=True, executable="/bin/bash", text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})  # in log ngay, không đợi hết
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh lỗi (exit {p.returncode}) — xem output ở trên")

# Mount Drive & GPU

In [2]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(OUT_DIR, exist_ok=True)
# không nạp BLIP-2 → GPU nào cũng được (T4/L4/A100)
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
name, memory.total [MiB]
NVIDIA L4, 23034 MiB


# Clone repo (nhánh EXP-004)

In [3]:
if not os.path.isdir("/content/Exp-CLIP"):
    !git clone -b $BRANCH https://github.com/huongng389/Exp-CLIP.git /content/Exp-CLIP
%cd /content/Exp-CLIP
assert os.getcwd() == "/content/Exp-CLIP", "clone lỗi — nhánh đã push chưa?"
!git checkout $BRANCH && git pull
!git log --oneline -1   # ghi lại commit vào research_log

Cloning into '/content/Exp-CLIP'...
remote: Enumerating objects: 401, done.
remote: Counting objects: 100% (401/401), done.
remote: Compressing objects: 100% (132/132), done.
remote: Total 401 (delta 293), reused 372 (delta 265), pack-reused 0 (from 0)
Receiving objects: 100% (401/401), 32.22 MiB | 20.50 MiB/s, done.
Resolving deltas: 100% (293/293), done.
/content/Exp-CLIP
Already on 'exp/004-text-anchor-loss'
Your branch is up to date with 'origin/exp/004-text-anchor-loss'.
Already up to date.
97771a1 (HEAD -> exp/004-text-anchor-loss, origin/exp/004-text-anchor-loss) EXP-004: text-anchor loss for the projection head


# Giải nén dữ liệu (train + test)

In [4]:
!mkdir -p /content/CAER-S_Face /content/static_test_sets
!unzip -q -n "$FACE_ZIP" -d /content/CAER-S_Face
!unzip -q -n "$TEST_ZIP" -d /content/static_test_sets
!ls /content/CAER-S_Face/train
!ls /content/static_test_sets

Angry  Disgust	Fear  Happy  Neutral  Sad  Surprise
AffectNet7  AffectNet8	FERPlus  RAFDB


# Cài dependencies (venv310)\ntorch 2.0.1 + lavis: dùng cho target, proto, weight, train, test.

In [5]:
!sudo apt-get -qq update > /dev/null
!sudo apt-get -qq install -y python3.10 python3.10-venv python3.10-distutils > /dev/null
!python3.10 -m venv /content/venv310
!/content/venv310/bin/pip install -q --upgrade pip
!/content/venv310/bin/pip install -q --no-cache-dir torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
!/content/venv310/bin/pip install -q --no-cache-dir transformers tqdm regex pyyaml pandas pydantic click timm einops
!/content/venv310/bin/pip install -q --no-cache-dir salesforce-lavis
# phải cài SAU lavis: lavis kéo numpy 2.x / opencv mới
!/content/venv310/bin/pip install -q --no-cache-dir "numpy<2" "opencv-python==4.10.0.84" "opencv-python-headless==4.10.0.84"
!/content/venv310/bin/pip install -q --no-cache-dir matplotlib scikit-learn
!/content/venv310/bin/python -c "import numpy, torch, lavis, sklearn; print('numpy', numpy.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 7.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 41.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Instal

# Cache feature CLIP + ngân hàng text
Feature ảnh dùng lại từ EXP-003. Ngân hàng text tính 1 lần.

In [6]:
if not os.path.exists(CLIP_CACHE):
    run(f'/content/venv310/bin/python tools/cache_clip_features.py --train-data {TRAIN_DATA} --out {CLIP_CACHE} '
        f'--load-model CLIP_L14 --batch-size 256 --workers 8')
if not os.path.exists(BANK):
    run(f'/content/venv310/bin/python tools/build_text_bank.py --variant {BANK_VARIANT} --out {BANK} --load-model CLIP_L14')
!ls -lh "$CLIP_CACHE" "$BANK"


  0%|                                               | 0.00/890M [00:00<?, ?iB/s]
  3%|█                                      | 24.0M/890M [00:00<00:03, 250MiB/s]
  5%|██                                     | 47.8M/890M [00:00<00:03, 235MiB/s]
  8%|███                                    | 70.3M/890M [00:00<00:08, 100MiB/s]
 10%|███▉                                   | 89.2M/890M [00:00<00:06, 121MiB/s]
 12%|████▋                                   | 106M/890M [00:00<00:06, 131MiB/s]
 14%|█████▌                                  | 125M/890M [00:00<00:05, 148MiB/s]
 17%|██████▋                                 | 149M/890M [00:01<00:04, 177MiB/s]
 20%|███████▉                                | 176M/890M [00:01<00:03, 204MiB/s]
 22%|████████▊                               | 197M/890M [00:01<00:04, 174MiB/s]
 24%|█████████▋                              | 216M/890M [00:01<00:04, 164MiB/s]
 26%|██████████▍                             | 233M/890M [00:01<00:05, 132MiB/s]
 28%|███████████▏          

# Train từng λ (30 epoch, từ cache)
Cấu hình như EXP-003, chỉ thêm loss neo. Bị ngắt thì chạy lại cell này: λ đã xong được bỏ qua, λ đang dở tự resume.

In [7]:
for lam in LAMBDAS:
    job = job_of(lam)
    if os.path.exists(f"{OUT_DIR}/{job}-e{SAVE_EPOCHS[-1]}-model.pth"):
        print(f"λ={lam}: đã train xong, bỏ qua"); continue
    resume = f"{OUT_DIR}/{job}-resume.pth"
    print(f"=============== λ = {lam}")
    run(f'/content/venv310/bin/python train.py '
        f'--epochs {EPOCHS} --batch-size 512 --batch-size-test-image 512 --batch-size-test-video 64 '
        f'--lr {LR} --weight-decay 1e-4 --momentum 0.9 --print-freq 1000 --milestones 1000 --seed {SEED} '
        f'--job-id {job} --instruction "{INSTRUCTION}" --load-model CLIP_L14 '
        f'--cache-clip {CLIP_CACHE} --cache-targets {PRE_DIR}/targets '
        f'--text-bank {BANK} --anchor-weight {lam} --anchor-type {ANCHOR} '
        f'--save-epochs {" ".join(map(str, SAVE_EPOCHS))} --extra-checkpoint-dir {OUT_DIR}'
        + (f' --resume {resume}' if os.path.exists(resume) else ''))

=============== λ = 0.1
Training date:  26-10-14 15:17
************************
workers = 8
epochs = 30
batch_size = 512
batch_size_test_image = 512
batch_size_test_video = 64
lr = 0.001
weight_decay = 0.0001
momentum = 0.9
print_freq = 1000
milestones = [1000]
seed = 1
job_id = exp004_anchor_crossA_l0.1_l14_seed1
instruction = Please play the role of a facial action describer. Objectively describe the detailed facial actions of the person in the image.
load_model = CLIP_L14
extra_checkpoint_dir = /content/drive/MyDrive/ExpCLIP_output/exp004
train_data = /data/EECS-IoannisLab/datasets/Static_FER_Datasets/CAERS_Face/train
sample_weights = None
cache_clip = /content/drive/MyDrive/ExpCLIP_output/exp001/full/clip_feats_l14.npz
cache_targets = /content/drive/MyDrive/ExpCLIP_output/exp001/full/targets
text_bank = /content/drive/MyDrive/ExpCLIP_output/exp004/text_bank_A_l14.npz
anchor_weight = 0.1
anchor_type = cross
save_epochs = [5, 10, 20, 30]
resume = None
************************
Cached 

# Test zero-shot từng mốc

In [8]:
!mkdir -p checkpoint log
for lam in LAMBDAS:
    for e in SAVE_EPOCHS:
        job = f"{job_of(lam)}-e{e}"
        if os.path.exists(f"{OUT_DIR}/{job}-test.txt"):
            !cp "$OUT_DIR/$job-test.txt" log/
            continue
        if not os.path.exists(f"checkpoint/{job}-model.pth"):
            !cp "$OUT_DIR/$job-model.pth" checkpoint/
        print(f"=============== λ = {lam}, epoch {e}")
        run(f'/content/venv310/bin/python test.py --load-model CLIP_L14 --job-id {job} '
            f'--data-root /content/static_test_sets 2>&1 | tee log/{job}-test.txt')
        !cp log/$job-test.txt "$OUT_DIR/"

=============== λ = 0.1, epoch 5
************************
load_model = CLIP_L14
job_id = exp004_anchor_crossA_l0.1_l14_seed1-e5
data_root = /content/static_test_sets
************************
************************************************************************** Zero-shot Prompt Type:  Class Name
******************** Static FER Zero-shot Performance ********************
************************* RAFDB
UAR/WAR: 50.99/54.82
************************* AffectNet7
UAR/WAR: 41.17/41.17
************************* AffectNet8
UAR/WAR: 35.58/35.58
************************* FERPlus
UAR/WAR: 40.87/44.61
************************************************************************** Zero-shot Prompt Type:  An Expression of Name
******************** Static FER Zero-shot Performance ********************
************************* RAFDB
UAR/WAR: 57.77/63.92
************************* AffectNet7
UAR/WAR: 44.46/44.46
************************* AffectNet8
UAR/WAR: 39.33/39.33
************************* FERPlus
U

# So sánh
Bảng 1: Ensemble Five (UAR) theo λ × epoch, λ = 0 là EXP-003. Bảng 2: Δ so với EXP-000 seed1. Bảng 3: đủ 5 prompt ở epoch 30, Δ so với EXP-000 seed1.

In [9]:
import re, pandas as pd
REF = pd.DataFrame([{"prompt": "Class Name", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 50.36, "WAR": 56.58}, {"prompt": "Class Name", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 42.51, "WAR": 42.51}, {"prompt": "Class Name", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 36.95, "WAR": 36.95}, {"prompt": "Class Name", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 38.27, "WAR": 45.45}, {"prompt": "An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 56.73, "WAR": 64.05}, {"prompt": "An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 44.0, "WAR": 44.0}, {"prompt": "An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 38.33, "WAR": 38.33}, {"prompt": "An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 39.99, "WAR": 52.92}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 54.94, "WAR": 65.12}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 43.97, "WAR": 43.97}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 38.02, "WAR": 38.02}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 36.63, "WAR": 48.89}, {"prompt": "Expression Ensemble Five", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 58.88, "WAR": 66.88}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 44.09, "WAR": 44.09}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 39.23, "WAR": 39.23}, {"prompt": "Expression Ensemble Five", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 40.65, "WAR": 51.47}, {"prompt": "Expression Ensemble Ten", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 57.73, "WAR": 65.91}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 44.94, "WAR": 44.94}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 39.55, "WAR": 39.55}, {"prompt": "Expression Ensemble Ten", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 40.45, "WAR": 49.29}, {"prompt": "Class Name", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 50.23, "WAR": 54.56}, {"prompt": "Class Name", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 41.47, "WAR": 41.47}, {"prompt": "Class Name", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 35.76, "WAR": 35.76}, {"prompt": "Class Name", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 38.58, "WAR": 45.82}, {"prompt": "An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 55.34, "WAR": 62.0}, {"prompt": "An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 43.79, "WAR": 43.79}, {"prompt": "An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 38.4, "WAR": 38.4}, {"prompt": "An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 38.89, "WAR": 51.62}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 55.64, "WAR": 64.78}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 44.39, "WAR": 44.39}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 38.08, "WAR": 38.08}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 37.95, "WAR": 49.17}, {"prompt": "Expression Ensemble Five", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 58.07, "WAR": 65.32}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 44.17, "WAR": 44.17}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 38.84, "WAR": 38.84}, {"prompt": "Expression Ensemble Five", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 41.55, "WAR": 53.41}, {"prompt": "Expression Ensemble Ten", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 57.19, "WAR": 64.6}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 44.97, "WAR": 44.97}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 39.45, "WAR": 39.45}, {"prompt": "Expression Ensemble Ten", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 42.17, "WAR": 49.46}, {"prompt": "Class Name", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "Class Name", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "Class Name", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "Class Name", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "An Expression of Name", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "An Expression of Name", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "An Expression of Name", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "An Expression of Name", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "Expression Ensemble Five", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "Expression Ensemble Five", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "Expression Ensemble Ten", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "Expression Ensemble Ten", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}])

def parse(path):
    rows, prompt, dataset = [], None, None
    for line in open(path):
        if "Prompt Type:" in line:
            prompt = line.split("Prompt Type:")[1].strip()
        elif re.match(r"^\*+ (RAFDB|AffectNet7|AffectNet8|FERPlus)$", line.strip()):
            dataset = line.strip().split()[-1]
        elif line.startswith("UAR/WAR:"):
            uar, war = map(float, line.split(":")[1].strip().split("/"))
            rows.append(dict(prompt=prompt, dataset=dataset, UAR=uar, WAR=war))
    return pd.DataFrame(rows)

res = pd.concat([parse(f"log/{job_of(lam)}-e{e}-test.txt").assign(lam=lam, epoch=e) for lam in LAMBDAS for e in SAVE_EPOCHS])
res.to_csv(f"{OUT_DIR}/exp004_anchor_{ANCHOR}{BANK_VARIANT}_l14_seed{SEED}-results.csv", index=False)
r3 = pd.read_csv(EXP003_CSV).assign(lam=0)
res = pd.concat([r3[r3.epoch.isin(SAVE_EPOCHS)], res])
base = REF[REF.src == "EXP-000 seed1"].set_index(["prompt", "dataset"])
pd.set_option("display.width", 250)
DS = ["RAFDB", "AffectNet7", "AffectNet8", "FERPlus"]

ens = res[res.prompt == "Expression Ensemble Five"].pivot_table(index=["lam", "epoch"], columns="dataset", values="UAR")[DS]
b5 = base.loc["Expression Ensemble Five"].UAR
display(pd.concat({"UAR": ens, "Δ vs EXP-000 seed1": (ens - b5).round(2)}, axis=1))

full = res[res.epoch == SAVE_EPOCHS[-1]].merge(base.UAR.rename("UAR_base").reset_index(), on=["prompt", "dataset"])
full["Δ"] = (full.UAR - full.UAR_base).round(2)
display(full.pivot_table(index=["prompt", "dataset"], columns="lam", values="Δ", sort=False))

UAR                               Δ vs EXP-000 seed1                              
dataset     RAFDB AffectNet7 AffectNet8 FERPlus              RAFDB AffectNet7 AffectNet8 FERPlus
lam  epoch                                                                                      
0.0  5      58.64      44.20      39.02   40.56              -0.24       0.11      -0.21   -0.09
     10     56.73      43.97      38.42   42.05              -2.15      -0.12      -0.81    1.40
     20     54.61      44.14      37.95   45.10              -4.27       0.05      -1.28    4.45
     30     53.68      44.20      37.85   44.96              -5.20       0.11      -1.38    4.31
0.1  5      60.11      44.49      39.38   42.05               1.23       0.40       0.15    1.40
     10     59.91      44.54      39.08   42.81               1.03       0.45      -0.15    2.16
     20     58.26      45.31      38.95   45.29              -0.62       1.22      -0.28    4.64
     30     57.76      45.66      39.27   48.13              -1.12       1.57       0.04    7.48
1.0  5      59.39      44.09      38.70   43.27               0.51       0.00      -0.53    2.62
     10     60.72      44.31      38.88   44.05               1.84       0.22      -0.35    3.40
     20     59.74      45.14      39.15   45.05               0.86       1.05      -0.08    4.40
     30     58.73      45.83      39.60   46.14              -0.15       1.74       0.37    5.49
10.0 5      58.32      43.26      38.50   42.74              -0.56      -0.83      -0.73    2.09
     10     58.77      43.89      38.80   43.88              -0.11      -0.20      -0.43    3.23
     20     59.07      44.23      39.17   45.79               0.19       0.14      -0.06    5.14
     30     57.52      44.26      39.20   45.67              -1.36       0.17      -0.03    5.02

lam                                                      0.0   0.1   1.0   10.0
prompt                                       dataset                           
Class Name                                   RAFDB       2.65  4.31  4.29  3.04
                                             AffectNet7  1.92  0.95  0.06 -1.60
                                             AffectNet8  0.38 -0.20 -0.03 -0.72
                                             FERPlus     4.80  6.92  4.95  3.79
An Expression of Name                        RAFDB      -3.39  1.54  1.11  0.50
                                             AffectNet7  0.49  1.49  0.86  0.00
                                             AffectNet8 -0.28  0.95  1.05  0.90
                                             FERPlus     6.13  7.43  5.92  5.00
A Photo of A Face with An Expression of Name RAFDB      -5.27  0.61  3.61  4.95
                                             AffectNet7 -1.91  0.03 -0.17 -1.00
                                             AffectNet8 -2.52 -0.64 -0.44 -1.29
                                             FERPlus     3.94  5.77  7.90  5.40
Expression Ensemble Five                     RAFDB      -5.20 -1.12 -0.15 -1.36
                                             AffectNet7  0.11  1.57  1.74  0.17
                                             AffectNet8 -1.38  0.04  0.37 -0.03
                                             FERPlus     4.31  7.48  5.49  5.02
Expression Ensemble Ten                      RAFDB      -5.05  0.12  1.13  0.16
                                             AffectNet7 -0.85  0.86  0.37 -0.11
                                             AffectNet8 -1.47  0.35 -0.22 -0.55
                                             FERPlus     3.67  5.26  6.41  4.62

# Log train
Accuracy train và loss neo trung bình ở mỗi mốc epoch.

In [10]:
import re
for lam in LAMBDAS:
    lines = open(f"{OUT_DIR}/{job_of(lam)}-log.txt").read().splitlines()
    acc = [float(l.split(":")[1]) for l in lines if l.startswith("The best accuracy")]
    anc = [float(re.search(r"Anchor [\d.]+ \(([\d.]+)\)", l).group(1)) for l in lines if "Anchor" in l]
    print(f"λ={lam}: acc", {e: round(acc[e - 1], 2) for e in SAVE_EPOCHS if e <= len(acc)},
          "| anchor (cuối mỗi lần in)", [round(a, 3) for a in anc[::max(1, len(anc) // 6)]])

λ=0.1: acc {5: 41.53, 10: 61.49, 20: 75.88, 30: 81.68} | anchor (cuối mỗi lần in) [0.16, 0.454, 0.361, 0.329, 0.294, 0.289]
λ=1: acc {5: 39.83, 10: 60.53, 20: 75.56, 30: 81.55} | anchor (cuối mỗi lần in) [0.16, 0.085, 0.062, 0.057, 0.051, 0.051]
λ=10: acc {5: 36.0, 10: 58.49, 20: 74.61, 30: 80.97} | anchor (cuối mỗi lần in) [0.16, 0.015, 0.013, 0.012, 0.011, 0.011]
